In [1]:
from openadmet.toolkit.database.chembl import PermissiveChEMBLTargetCurator
from openadmet.toolkit.chemoinformatics.rdkit_funcs import canonical_smiles, smiles_to_inchikey
from tqdm.auto import tqdm
tqdm.pandas()
import datamol as dm

/Users/cynthiaxu/miniconda3/envs/openadmet/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Curating basic pChEMBL data and pushing to a remote intake catalog

Our goal is to curate activity data from ChEMBL and push this to a remote location with a catalog that can be used by others to look up our data. This will enable consistency and rapid dissemination of our work as well as an over-time evolution of our data curation practices. 

We use the `Intake` package for a lightweight self-describing data parsing workflow. Read more about intake here: https://intake.readthedocs.io/en/latest/index.html


Here we gather `pChEMBL` data permissivley from ChEMBL (ie without activity based curation) for our 5 main targets (AHR, PXR, CYP3A4, CYP2C9, CYP2D6) and also additional target CYP2J2.

We then aggregate `pChEMBL` measurements on the same compound by taking the mean. This is the most basic form of curation available, but serves as a good baseline for our initial models. 


## gather ChEMBL data

First we need to gather in our data from ChEMBL using our SQL API defined in `openadmet-toolkit`

We use `OPENADMET_CANONICAL_SMILES` and `OPENADMET_INCHIKEY` to distinguish our ML ready representation from the source SMILES

In [2]:
def gather_chembl_data_for_target(target_name: str, chembl_tid: str, chembl_ver: int):
    print(f"working on target {target_name}")
    pctc = PermissiveChEMBLTargetCurator(chembl_target_id=chembl_tid, version=chembl_ver)
    activity_data = pctc.get_activity_data(return_as="df")
    print("canonicalising raw data")
    with dm.without_rdkit_log():
        activity_data["OPENADMET_CANONICAL_SMILES"] = activity_data["canonical_smiles"].progress_apply(lambda x: canonical_smiles(x))
        activity_data["OPENADMET_INCHIKEY"] = activity_data["OPENADMET__CANONICAL_SMILES"].progress_apply(lambda x: smiles_to_inchikey(x))
    # important to canonicalise here so compound deduplication is done correctly
    aggregated_activity = pctc.aggregate_activity_data_by_compound(canonicalise=True)
    print("smiles duplicates", aggregated_activity["OPENADMET_CANONICAL_SMILES"].duplicated().sum())
    print("inchikey duplicates", aggregated_activity["OPENADMET_INCHIKEY"].duplicated().sum())
    return aggregated_activity, activity_data
        

## Define target metadata

We need the CHEMBL codes for our targets

In [3]:
targets = {
    "AHR": "CHEMBL3201",
    "PXR": "CHEMBL3401",
    "CYP1A2": "CHEMBL3356",
    "CYP3A4": "CHEMBL340",
    "CYP2C9": "CHEMBL3397",
    "CYP2D6": "CHEMBL289",
    "CYP2J2": "CHEMBL3491"
}

In [4]:
chembl_ver = 35

In [5]:
from openadmet.toolkit.webservices.credentials import S3Settings
from openadmet.toolkit.webservices.s3 import S3Bucket

# Setup S3

After curating our data we would like to push to a remote bucket to save both the raw data and the catalog

In [6]:
settings = S3Settings()

ValidationError: 2 validation errors for S3Settings
AWS_ACCESS_KEY_ID
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
AWS_SECRET_ACCESS_KEY
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing

In [ ]:
bucket = "openadmet-data-public-dev"

In [ ]:
bucket = S3Bucket.from_settings(settings, bucket)

In [ ]:
import datetime

In [ ]:
t = datetime.datetime.now()

In [ ]:
date = t.strftime("%Y-%m-%d")

In [ ]:
location=f"ChEMBL{chembl_ver}_permissive_{date}"

In [ ]:
import os
from pathlib import Path

location_path = Path(location)

In [ ]:
location_path.mkdir(exist_ok=False)

# Main loop

Generate the data for each target and save to parquet, then push to S3 data lake with parquet files. 

We use parquet here for improved performance and reduced size on disk.

In [ ]:
uris_raw = {}
uris_agg = {}
for target, chembl_tid in targets.items():
    agg, raw  = gather_chembl_data_for_target(target, chembl_tid, chembl_ver)
    # TODO: make a function this is clunky
    fname_agg = f"ChEMBL_permissive_{target}_{chembl_tid}_aggregated.parquet"
    fname_raw = f"ChEMBL_permissive_{target}_{chembl_tid}_raw.parquet"
    
    agg.to_parquet(location_path/fname_agg)
    raw.to_parquet(location_path/fname_raw)
    
    bucket_destination_agg = location + "/" + fname_agg
    bucket.push_file(location_path/fname_agg, bucket_destination_agg)
    bucket_destination_raw = location + "/" + fname_raw
    bucket.push_file(location_path/fname_raw, bucket_destination_raw)

    # get S3 URIs
    uri_agg = bucket.to_uri(bucket_destination_agg)
    uris_agg[target] = uri_agg

    uri_raw = bucket.to_uri(bucket_destination_raw)
    uris_raw[target] = uri_raw

    


working on target AHR
canonicalising raw data


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 310/310 [00:00<00:00, 4943.12it/s]


smiles duplicates 0
inchikey duplicates 0
working on target PXR
canonicalising raw data


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1340/1340 [00:00<00:00, 3674.56it/s]


smiles duplicates 0
inchikey duplicates 0
working on target CYP1A2
canonicalising raw data


...

smiles duplicates 0
inchikey duplicates 0
working on target CYP2D6
canonicalising raw data


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12281/12281 [00:02<00:00, 4165.18it/s]


smiles duplicates 0
inchikey duplicates 0
working on target CYP2J2
canonicalising raw data


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 67/67 [00:00<00:00, 3829.11it/s]


smiles duplicates 0
inchikey duplicates 0


# Build the Intake Catalog

We have sucessfully aggregted our data and pushed it to a remote destination. Now for others to consume our data, we are going to make an `Intake` catalog such that our data can be readily made available. 

The workflow here is drawn from the `creator` walkthrough from the main intake tutorials https://intake.readthedocs.io/en/latest/walkthrough2.html

TODO: add descriptions to the catalog

In [ ]:
import intake

ModuleNotFoundError: No module named 'intake'

In [ ]:
intake.Catalog?

In [ ]:
cat = intake.entry.Catalog()

In [ ]:
uris_agg

In [ ]:
uris_raw

In [ ]:
for k,v in uris_agg.items():
    cat[k+"_aggregated"] = intake.readers.PandasParquet(v)

In [ ]:
for k,v in uris_raw.items():
    cat[k+"_raw"] = intake.readers.PandasParquet(v)

## Push the Catalog

Ok now we have made the catalog, lets push it to the remote location so it can live alongside the data. 

The catalog can then be used from S3 or from github etc, anything that exposes a file-like API. 

In [ ]:
cat

In [ ]:
catname = f"CATALOG_{location}.yaml"

In [ ]:
cat.to_yaml_file(catname)

In [ ]:
cat_location = location+ "/" +catname

In [ ]:
cat_location

In [ ]:
bucket.push_file(catname, cat_location)

In [ ]:
cat_uri = bucket.to_uri(cat_location)

In [ ]:
# Now can read the catalog from URI
# cat = intake.Catalog.from_yaml_file("s3://openadmet-data-public-dev/ChEMBL34_permissive_2025-02-12/CATALOG_ChEMBL34_permissive_2025-02-12.yaml")